# 自己回帰型ニューラルネットワークと時系列処理

PyTorchによる自己回帰型ニューラルネットワーク（Recurrent Neural Network：RNN）の記述方法を学習します．  
今回は長短期記憶（Long Short Term Memory：LSTM）を持ったRNNの記述方法を学習します．  

**目標：RNNとMLPの違いを理解**

---

例題では，回帰問題（気温予測）を解きます．  
演習では，分類問題（季節分類）を解きます．

※rnn_data.csvというデータセットを使います  
　2015〜2020年の日中平均気温と日付が記録されたデータセットです

---
## この教材について

「3分で学ぶPyTorch」シリーズの **RNN（自己回帰型ニューラルネットワーク） 基礎編（第1回）** の演習パートです。

このノートブックは**回答ファイル（ans）**です。全演習の解答が入っています。まずは演習ファイル（task）に挑戦してから参照してください。

この教材と関連記事は note で無料公開しています。
シリーズ一覧: https://note.com/technosend/m/m84d841b6d067

---

## rnn_data.csvデータセット
- 2015〜2020年の日中平均気温と日付が記録されたデータセット
  - 第1カラム：日付
  - 第2カラム：日中の平均気温
- 全部で2192個
  365*6+2（2016年と2020年が閏年なため）
- 学習データ：先頭1826個（2016年から2019年までのデータ）
- テストデータ：後ろから366個（2020年のデータ）
- 値域：-2.0~32.8  
  最低平均気温が-2℃で最高平均気温が32.8℃
- 例題：気温予測での使い方  
  - 50日分のデータを1つの時系列として学習させる  
  - 今の日付に対して1日後を教師データとする  
  （ = 入力データ1つにつき，1つの教師データがある）
  - データの形状は50, 1776, 1になる  
  （ = 50日で1セットのデータが1776個分ある）
  - テスト時には365日分のデータを1セットにして入力して出力させる  
  ※本来であれば50日を1セットにした方がいいが，プログラムの簡単化のためこの仕様にしている
- 演習：季節分類での使い方
  - 50日分のデータを1つの時系列として学習させる 
  - 50日分のデータに対して1つの季節分類を教師データとする  
  （ =49日分の出力は無視して50日目の出力を分類結果とする）
    - 季節分類は日付のデータを次の規則で変換させる
      - 3〜5月：春
      - 6〜8月：夏
      - 9〜11月：秋
      - 12〜2月：冬
  - データの形状は50, 1776, 1になる  
  （ = 50日で1セットのデータが1776個分ある）
  - テスト時には365日分のデータを1セットにして入力して出力させる  
  ※本来であれば50日で1セットにして49日分無視して50日目のデータを取得する方がいいが，  
  出力の見栄えとプログラムの簡単化のためこの仕様にしている


## 例題 気温予測

**LSTM1層・全結合1層のRNNの作成**  
1日の平均気温を時系列データとして順次入力し，次の日の平均気温を出力する回帰問題を解く．

<!-- 
- 2015〜2019年のデータを使用
- 入力は28日（4週間）の日平均気温，出力はその次の日の日平均気温
  - ex：2015/1/1〜2015/1/28の日平均気温が入力 → 2015/1/29の日平均気温が出力 -->

### 例題1. ライブラリのインポート

深層学習演算ライブラリPyTorchなどのライブラリ，パッケージ，モジュールをインポートする．

---


<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=14XdT7XWs6JzTil6JhZZw7aM3VpAxdYH2&sz=w400">




#### 例題1のコード

In [ ]:
# 例題1. ライブラリのインポート

import torch
import torch.nn as nn
import torch.optim as optim
 
import numpy as np
import matplotlib.pyplot as plt
# japanize-matplotlib は古いパッケージです。
# 代替: !pip install matplotlib-fontja -q  (2023年以降推奨)
!pip install japanize-matplotlib -q
import japanize_matplotlib
!pip install gdown --upgrade -q  # gdown 4.6以降はアップグレード推奨
import gdown

#### 今回使うライブラリ，パッケージ，モジュール一覧

- [torch](https://pytorch.org/docs/stable/torch.html)：多次元テンソルのデータ構造とそのテンソルのための算術演算が組み込まれたパッケージ
- [torch.nn](https://pytorch.org/docs/stable/nn.html)：ニューラルネットワークを定義するためのパッケージ  
nnという略称を与えることが多い
- [torch.optim](https://pytorch.org/docs/stable/optim.html)：最適化器を宣言するためのパッケージ  
optimという略称を与えることが多い
- [numpy](https://numpy.org/doc/stable/user/whatisnumpy.html)：行列演算を行うライブラリ（今回は画像の表示のために使用）  
npという略称を与えることが多い
- [matplotlib.pyplot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.html)：グラフや画像の描画を行うモジュール  
pltという略称を与えることが多い
- [japanize_matplotlib](https://github.com/uehara1414/japanize-matplotlib)：Matplotlibで日本語を表示するためのライブラリ
- [gdown](https://github.com/wkentaro/gdown)：GoogleDriveからファイルをダウンロードするためのライブラリ

<font color="blue">【TASK】</font>パッケージをインポートしましょう

### 例題2. ニューラルネットワークの定義

ニューラルネットワーククラスを定義して，そのクラスのインスタンスを宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1jl1W_RW7HrU0ksk1a0XrSq6CyldXF4qZ&sz=w400">

#### 例題2のコード

In [ ]:
# 例題1.2. ニューラルネットワークの定義

# 1. ニューラルネットワーククラスの定義
class TempPredictor(nn.Module):
    def __init__(self):
        super(TempPredictor, self).__init__()
        self.lstm1 = nn.LSTM(1, 10, num_layers=1, batch_first=False)
        self.fc1 = nn.Linear(10, 1)
        
    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.fc1(x)
        return x
 
# 2. インスタンスの宣言
temp_predictor = TempPredictor()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
temp_predictor.to(device)

#### 1. ニューラルネットワーククラスの定義  

- nn.Moduleを継承したクラス「TempPredictor」を定義
- \_\_init\_\_()とforward()を定義
  ```python
  class TempPredictor(nn.Module):
    def __init__(self):
        super(TempPredictor, self).__init__()
        # 【TASK】LSTM層と全結合層を宣言
    def forward(self, x):
        # 【TASK】順伝播のパスを定義
        return x
  ```


- \_\_init\_\_()では，LSTM層と全結合層を宣言
  - [super()](https://docs.python.org/ja/3/library/functions.html#super)を呼び出す
  - 一つの層につき一つ，nnパッケージ内のクラスのインスタンスを宣言するので，今回は二つ宣言
  - **<font color="red">【NEW!】</font>**LSTM層は[nn.LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)クラスを用いて宣言
  ```python
  self.lstm1 = nn.LSTM(1, 10, num_layers=1, batch_first=False)
  # 第１引数：入力次元数（int）
  # 第２引数：出力次元数（int）
  # 第３引数（オプション）：レイヤー数（int），デフォルト1
  # 第4引数（オプション）：バッチファースト(bool)，デフォルトFalse
  ```
  - 全結合層は[nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear)クラスを用いて宣言  
  ```python
  self.fc1 = nn.Linear(10, 1)
  ```
  <font color="blue">【TASK】</font>LSTM層と全結合層をnn.LSTMクラスとnn.Linearクラスを用いて宣言しましょう  
  ニューラルネットワークの構成は下記の通りです
      - LSTM層1：入力1，出力10，レイヤー数1
      - 全結合層1：入力10，出力1

- forward()では，順伝播のパスを定義  
  - forward()は外部からの入力データxを受け取り，順伝播を行う
  - nn.Linearクラスのインスタンスに()をつけて引数を与えて呼び出すと，クラスメソッドである\_\_call_\_()が呼び出され，引数に対する全結合層の計算結果が返される
  - <font color="red">【NEW!】</font>nn.LSTMクラスのインスタンスに()をつけて引数を与えて呼び出すと，クラスメソッドである\_\_call_\_()が呼び出され，引数に対するLSTM層の計算結果が返される  
  ※nn.LSTMクラスの\_\_call_\_()の戻り値はLSTMの出力とセル状態の二つあるが，今回はLSTMの出力のみ取得
  ```python
  x, _ = self.lstm1(x)
  x = self.fc1(x)
  ```
  <font color="blue">【TASK】</font>順伝播のパスを定義しましょう  
  時系列データ（数値）を入力すると想定したとき順伝播のパスの構成は下記の通りです
      1. LSTM層1  
      2. 全結合層1  

    ※PyTorchのLSTM層は内部に活性化関数を持つので，今回は2つの層の間に活性化関数がありません．

#### 2. インスタンスの宣言

- temp_predictorという名前でインスタンスを宣言
```python
temp_predictor = # 【TASK】ニューラルネットワーククラスのインスタンスの宣言
```

- GPUにセットアップ
```python
device = # 【TASK】GPUの指定
# 【TASK】GPUにセットアップ
```

- ニューラルネットワーククラスのインスタンスの宣言

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスを宣言しましょう 

- GPUにセットアップ
  - Moduleクラスにもto()がある
  - Tensor型変数同様にto()を使ってGPUにデータを渡すことができる
  - 学習時にGPUを使う場合は，ニューラルネットワーククラスのインスタンスをGPUに渡す

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスをGPUにセットアップしましょう  
  - デバイス指定用の変数deviceを宣言しましょう
  - `torch.cuda.is_available()` でGPU有無を判定し，利用可能な場合は"cuda:0"，利用不可の場合は"cpu"を指定しましょう
    - [torch.device](https://pytorch.org/docs/stable/tensor_attributes.html#torch.torch.device)クラスを使ってデバイスを指定しましょう
  - [to()](https://pytorch.org/docs/1.9.1/generated/torch.Tensor.to.html)を使ってGPUにセットアップしましょう


### 例題3. 誤差関数・最適化器の設定

誤差関数と最適化器を宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1oS8_oitSvcQ9f5_uYMrDXyC3P_vrM7jp&sz=w400">



#### 例題3のコード

In [ ]:
# 例題3. 誤差関数・最適化器の設定

# 1. 誤差関数の宣言
criterion = nn.MSELoss()

# 2. 最適化器の宣言
optimizer_temp_predictor = optim.SGD(temp_predictor.parameters(), lr=0.01)

#### 1. 誤差関数の宣言  
  - 平均二乗誤差（回帰問題のため）を計算するcriterionを宣言

    ```python
    criterion = # 【TASK】誤差関数の宣言
    ```



  - 平均二乗誤差は[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)クラスを用いて宣言
  - nn.MSELoss()で宣言したインスタンスは，二つの引数の平均二乗誤差を返す  

  ```python
  criterion = nn.MSELoss()
  ```
  
<font color="blue">【TASK】</font>誤差関数を宣言しましょう
- 平均二乗誤差関数を使いましょう
- nn.MSELossクラスを用いて宣言しましょう

#### 2. 最適化器の宣言
  - 確率的勾配降下法を計算するoptimizer_temp_predictorを宣言

    ```python
    optimizer_temp_predictor = # 【TASK】最適化器の宣言
    ```

  - 確率的勾配降下法は[optim.SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html#torch.optim.SGD)クラスを用いて宣言
  ```python
  optimizer_temp_predictor = optim.SGD(temp_predictor.parameters(), lr=0.01)
  # 第１引数：ニューラルネットワークのパラメータ
  # 第２引数：学習率（float）
  ```

<font color="blue">【TASK】</font>最適化器を宣言しましょう  
- 確率的勾配降下法を使いましょう
- optim.SGDクラスを用いて宣言しましょう
- 引数の構成は下記の通りです  
  - ニューラルネットワークのパラメータtemp_predictorのパラメータ
  - lr：学習率0.01

### 例題4. データセットの準備

自作した関数make_dataset()を使ってデータセットを作成する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=17fS-oMI83rSL7SxN_GyKHjxP-FO-R3aQ&sz=w400">


#### 例題4のコード

まずはgdown.download()を使ってファイルをダウンロードします．
```python
gdown.download(id=file_id, output="rnn_data.csv", quiet=True)
# id：Google DriveのファイルID(str)  ※ gdown 4.6以降の推奨形式
# output（オプション）：ダウンロード時のファイル名(str)
# quiet（オプション）：ダウンロードのログを出力するかどうか(bool)
```

<font color="blue">【TASK】</font>セルを実行してファイルをダウンロードしましょう

In [ ]:
# データのダウンロード
# 旧形式URL変換済み: uc?export=download は gdown 4.6以降で不安定
file_id = "1ShNdssPwqjcMXgI_z_HDNI7ei6oLimGu"  # Google Drive ファイルID
gdown.download(id=file_id, output="rnn_data.csv", quiet=True)

In [ ]:
# 例題4. データセットの準備 

# 1. データセットを作成する関数の用意
def get_data(data, start, seq_len=50):
    arr = []
    for t in range(start, seq_len+start):
        arr.append([[data[t+i]] for i in range(len(data) - seq_len)])

    # list化しているのでnumpy.ndarray型に戻して返す
    return np.array(arr)

def make_dataset(data, train=True, seq_len=50):
    # train=True：学習データの作成
    if train:        
        # 入力データと教師データに分割
        inputs = get_data(data, start=0, seq_len=seq_len)
        targets = get_data(data, start=1, seq_len=seq_len)

        # torch.tensorへ変換
        inputs = torch.tensor(inputs, dtype=torch.float32) 
        targets = torch.tensor(targets, dtype=torch.float32) 
        
    # train=False：テストデータの作成
    else:
        # 入力データと教師データに分割
        inputs = data[:-1]
        targets = data[1:]
        
        # torch.tensorへ変換
        inputs = torch.tensor(inputs, dtype=torch.float32).view(inputs.shape[0], 1, 1)
        targets = torch.tensor(targets, dtype=torch.float32).view(targets.shape[0], 1, 1)
  
    print("入力データの次元数 :", inputs.shape)
    print("教師データの次元数 :", targets.shape)
    return [[inputs, targets]]

# 2. データの読み込み
temp_data= np.loadtxt("./rnn_data.csv", delimiter=",", usecols=1)
print(min(temp_data))
print(max(temp_data))
# 3. 値域を0〜1の範囲に正規化
temp_data = temp_data / 40

# 4. データセットの作成
seq_len = 50
print("データセットの概要")
print("学習データ")
train_loader = make_dataset(temp_data[:-366], train=True, seq_len=seq_len)
print("テストデータ")
test_loader = make_dataset(temp_data[-366:], train=False)

#### 1. データセットを作成する関数の用意

- get_data()：1時系列あたりのデータ長を指定すると分割して返す関数

  ```python
  inputs = get_data(data, start=0, seq_len=seq_len)
  # 第1引数：気温データ
  # 第2引数：取得を開始するインデックス，入力データと教師データの時刻をずらすために使う(int)，デフォルトTrue
  # 第3引数（オプション）：時系列データの系列長(int)，デフォルト50
  # 戻り値1：時系列データの系列長×バッチ数×各時刻のデータになっている配列（numpy.ndarray型）
  ```

- make_dataset()：気温データと予測する次の日の気温をデータセット化する関数
  
    ```python
    make_dataset(data, train=True, seq_len=50)
    # 第1引数：気温データ
    # 第2引数（オプション）：学習データを作成するか指定(bool)，デフォルトTrue
    # 第3引数（オプション）：時系列データの系列長(int)，デフォルト50
    # 戻り値1：入力データと教師データの配列(list)
    ```

#### 2. データの読み込み 

- temp_dataという名前で気温データ用の変数を宣言
- **<font color="red">【NEW!】</font>**[np.loadtxt()](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html)を用いてcsvファイルを読み込む

  ```python
  temp_data = np.loadtxt("./rnn_data.csv", delimiter=",", usecols=1)
  # 第1引数：ファイルパス (str)
  # 第2引数（オプション）：データを分割するための文字を指定 (str)
  # 第3引数（オプション）：使う列番号(0, 1, ...)を指定 (int)
  ```



#### 3. 値域を0〜1の範囲に正規化

- 気温データの値域を正規化
- 正規化の結果はtemp_dataに格納
- **<font color="red">【NEW!】</font>**nn.LSTMクラスの出力は値域が-1〜1になるため，事前にデータの最大値が1になるように調整

  ```python
  temp_data = temp_data / 40
  ```


#### 4. データセットの作成
- seq_lenという名前でデータ長用の変数を宣言
- train_loaderという名前で学習データのデータローダー用の変数を宣言
- test_loaderという名前でテストデータのデータローダー用の変数を宣言  
  
  ```python
  seq_len = # データ長
  train_loader = # 学習データのデータローダー
  test_loader = # テストデータのデータローダー
  ```



  - make_dataset()を使ってデータセットを作成
    - nn.LSTMクラスでは入力するデータを[データ長 x バッチサイズ x 各時刻の入力サイズの3次元配列にする必要がある](https://drive.google.com/file/d/1awq8ybxQKjkOLQcftn1V_HSXsJ3m08ic/view?usp=sharing)
    - ここでは2015〜2019年を学習用，2020年をテスト用のデータとする
    - 2020年は366日あるので，気温データの後ろ366個をテスト用，残りを学習用のデータとして分割
      
  ```python
  train_loader = make_dataset(temp_data[:-366], train=True, seq_len=seq_len)
test_loader = make_dataset(temp_data[-366:], train=False)
  ```



#### おまけ：データの可視化


In [ ]:
# おまけ：データの可視化
plt.plot(train_loader[0][0][:, 0, 0] * 40, label="入力データ")
plt.plot(train_loader[0][1][:, 0, 0] * 40, label="教師データ")
plt.ylabel("気温 [℃]")
plt.xlabel("時刻 （日にち）")
plt.legend()
plt.show()

- グラフの描画
  - 散布図の描画には[plt.plot()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)を使う
    ```python
    plt.plot(train_loader[0][0][:, 0, 0] * 40, label="入力データ")
    plt.plot(train_loader[0][1][:, 0, 0] * 40, label="教師データ")
    # 第1引数：x軸の値(list)
    # 第2引数：y軸の値(list)
    # 第3引数：凡例(str)
    ```

- 軸のラベルを設定
    - y軸のラベルの設定には[plt.ylabel()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.ylabel.html#matplotlib-pyplot-ylabel)を使う
    - x軸のラベルの設定には[plt.xlabel()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xlabel.html#matplotlib-pyplot-xlabel)を使う
    ```python
    plt.ylabel("気温 [℃]")
    plt.xlabel("時刻 （日にち）")
    # 第1引数：ラベル(str)
    ```

- 凡例の表示
    - 凡例の表示は[plt.legend()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html#matplotlib-pyplot-legend)を使う
    - 凡例の内容はplt.plot()の引数であるlabelで事前に設定しておく
    ```python
    plt.legend()
    ```

- グラフの表示
    - 描画したグラフの表示には[plt.show()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.show.html#matplotlib-pyplot-show)を使う
    ```python
    plt.show()
    ```


<font color="blue">【TASK】</font>実行してデータを可視化してみましょう

### 例題5. 学習

教師データとの誤差を計算し，パラメータを更新する．  
<!-- 学習時の誤差とテスト時の誤差を表示し，テスト時の予測精度を表示する．  -->

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1A6TjumoBejWpKGvD6TDQeivN1tpeEBp4&sz=w400">

#### 例題5のコード：前半の学習部分

In [ ]:
# 例題5. 学習

# 1. 学習ループの作成
epochs = 1000

# エポックのループ
for epoch in range(epochs):
    # 学習データのデータローダーのループ
    for data in train_loader:

        # 2. ニューラルネットワークへのデータの入力
        inputs, targets = data
        inputs = inputs.to(device)
        targets = targets.to(device)
        outputs = temp_predictor(inputs)
    
        # 3. 誤差の計算（BPTTパターン1：全ての時刻に教師データがある場合）
        loss = torch.tensor(0.0).to(device)
        for i in range(seq_len):
            loss += criterion(outputs[i], targets[i])
        # あるいはこの書き方でも大丈夫です
        # loss = criterion(outputs, targets) * seq_len
        
        # 4. 誤差逆伝播とパラメータの更新
        optimizer_temp_predictor.zero_grad()
        loss.backward()
        optimizer_temp_predictor.step()
    
        # 現在の誤差の値の表示
        if (epoch + 1) % 100 == 0:
            print("epoch :", epoch + 1, "学習誤差 :",loss.item())
    

#### 1. 学習ループの作成  
- ミニバッチ学習を行う学習ループの作成
- epochsという名前でエポック用の変数を宣言
- 外側にエポック，内側に学習データのデータローダーのループを作成
```python
epochs = # 【TASK】エポック数
for epoch in # 【TASK】エポック
      for data in # 【TASK】学習データのデータローダー
```

<font color="blue">【TASK】</font>学習ループを作成しましょう  
ループの設定は下記の通りです  
  - エポック
    - エポック数1000
    - [range](https://docs.python.org/ja/3/library/stdtypes.html#range)クラスを使ってループさせましょう
  - 学習データのデータローダー
    - ループ対象：学習データのデータローダーtrain_loader
    - [for](https://docs.python.org/ja/3/reference/compound_stmts.html#for)を使ってtrain_loaderをループさせましょう

#### 2. ニューラルネットワークへのデータの入力  

- GPUにセットアップ
- outputsという名前で出力用の変数を宣言
```python
inputs, labels = data
inputs = # 【TASK】GPUにセットアップ
targets = # 【TASK】GPUにセットアップ
outputs = # 【TASK】ニューラルネットワークからの出力
```


  
<font color="blue">【TASK】</font>ニューラルネットワークへデータを入力しましょう
  - 入力inputsをGPUにセットアップしましょう
  - 教師データtargetsをGPUにセットアップしましょう
  - inputsをニューラルネットワークに与えて出力outputsを取得しましょう

#### 3. **<font color="red">【NEW!】</font>** 誤差の計算（BPTTパターン1：全ての時刻に教師データがある場合）

- loss という名前で誤差計算の結果用の変数を宣言 
- データ長だけ繰り返すforループの作成

  ```python
  loss = # 【TASK】0に初期化
  for i in # 【TASK】データ長
      loss += # 【TASK】誤差計算
  ```



  - lossという変数をtorch.tensorクラスのオブジェクトとして宣言し，0.0で初期化
  - GPUにセットアップ
  ```python
  loss = torch.tensor(0.0).to(device)
  ```

- 誤差計算はnn.MSELossクラスやnn.CrossEntropyLossクラスなどの誤差関数クラスのインスタンスに引数を2つ与えて行う
```python
loss = criterion(outputs, labels)
# 第1引数：ニューラルネットワークの出力
# 第2引数：教師データ
```

  - 全時刻分足し合わせるので，forループを使って各時刻ごとの誤差を計算しlossに足す  
  ループ回数はデータ長だけ繰り返す
  ```python
  for i in range(seq_len):
        loss += criterion(outputs[i], targets[i])
  ```

<font color="blue">　【TASK】</font>誤差の計算（全ての時刻に教師データがある場合）を行いましょう  
ループの設定は以下の通りです．
  - データ長：seq_len
  - rangeを使ってseq_lenと同じ要素数の配列を生成して，ループさせましょう
    

#### 4. 誤差逆伝播とパラメータの更新  

- パラメータの微分値を初期化
- 誤差逆伝播
- パラメータの更新
```python
# 【TASK】パラメータの微分値を初期化
# 【TASK】誤差逆伝播
# 【TASK】パラメータの更新
```


- パラメータの微分値の初期化は最適化器が持つ
  [zero_grad()](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html#torch.optim.Optimizer.zero_grad)を呼び出して行う
  
- 誤差逆伝播は計算結果を持った変数から[backward()](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html)を呼び出して行う
- パラメータの更新は最適化器の持つ[step()](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html#torch.optim.Optimizer.step)を呼び出して行う  

<font color="blue">　【TASK】</font>誤差逆伝播とパラメータの更新を行いましょう
- zero_grad()を使ってパラメータの微分値の初期化をしましょう
- backward()を使って誤差逆伝播させましょう
- step()を使ってパラメータの更新を行いましょう





#### 例題5のコード：後半の予測部分

In [ ]:
# 5. 気温の予測

# ニューラルネットワークへのデータの入力
inputs, targets = test_loader[0]
inputs = inputs.to(device)
outputs = temp_predictor(inputs)

# データの加工
outputs = (outputs * 40).reshape(365).tolist()
# outputs = outputs.to('cpu').detach().numpy().copy()
targets = (targets * 40).reshape(365)

# グラフの描画
plt.plot(outputs, label="予測結果")
plt.plot(targets, label="教師データ")
plt.ylabel("気温 [℃]")
plt.xlabel("時刻 （日にち）")
plt.legend()
plt.show()

#### 5. 気温の予測
  - テストデータを入力したときの出力から予測結果を取得
  ```python
  inputs = # 【TASK】GPUにセットアップ
  outputs = # 【TASK】ニューラルネットワークからの出力

  outputs = # 【TASK】データの加工
  test_target = (test_target * 40).reshape(365)

  # 【TASK】予測結果を描画
  # 【TASK】正解データを描画
  # 【TASK】y軸のラベルを設定
  # 【TASK】x軸のラベルを設定
  plt.legend()
  plt.show()
  ```

- データの加工
    - 各時刻の出力が格納されていて（1, 365）という形状をしているので(365)とまとまるように並べ替える  
    - Tensor型データから複数のデータを抜き出す場合は[tolist()](https://pytorch.org/docs/stable/generated/torch.Tensor.tolist.html)を使う  
    ※tensor.tolist()と同じ動きをtensor.[detach()](https://pytorch.org/docs/stable/generated/torch.Tensor.detach.html#torch.Tensor.detach).[numpy()](https://pytorch.org/docs/stable/generated/torch.Tensor.numpy.html#torch.Tensor.numpy)で行うこともできる  
    ※データセット作成時に0〜1になるように正規化しているので元に戻している
    ```python
    outputs = (outputs * 40).reshape(365).tolist()
    # outputs = outputs.to('cpu').detach().numpy().copy()
    test_target = (test_target * 40).reshape(365)
    ```

- グラフの描画
    - 散布図の描画にはplt.plot()を使う
    ```python
    plt.plot(outputs, label="予測結果")
    plt.plot(test_target, label="教師データ")
    # 第1引数：x軸の値(list)
    # 第2引数：y軸の値(list)
    # 第3引数：凡例(str)
    ```

    - 軸のラベルを設定
        - y軸のラベルの設定にはplt.ylabel()を使う
        - x軸のラベルの設定にはplt.xlabel()を使う
        ```python
        # 第1引数：ラベル(str)
        plt.ylabel("気温 [℃]")
        plt.xlabel("時刻 （日にち）")
        ```

    - 凡例の表示
        - 凡例の表示はplt.legend()を使う
        - 凡例の内容はplt.plot()の引数であるlabelで事前に設定しておく
        ```python
        plt.legend()
        ```

    - グラフの表示
        - 描画したグラフの表示にはplt.show()を使う
        ```python
        plt.show()
        ```
 
<font color="blue">　【TASK】</font>テストデータを使って気温を予測して，その結果をグラフ描画しましょう  
- ニューラルネットワークへテストデータを入力し，気温予測を行いましょう
- グラフを描画しましょう  
    グラフの設定は下記の通りです  
    - 予測結果と正解データ（テストデータの教師データ）のために2系列描画します
    - 予測結果はoutputsに，正解データはtest_targetにそれぞれ格納されています
    - 両方のデータを加工しましょう
    - x軸のデータは日にち：365
    - y軸のデータは予測結果と正解データ
    - y軸のラベルは"気温 [℃]"，x軸のラベルは"時刻 （日にち）"